In [1]:
import re
import logging

logger = logging.getLogger("TEV")
logging.basicConfig(
    format="%(asctime)s %(levelname)-8s [%(name)s] %(message)s",
    filename= "output/count_gates.log",
    encoding="utf-8",
    level=logging.INFO,
    datefmt="%Y-%m-%d %H:%M:%S",
)
logging.getLogger('crsq').setLevel(logging.INFO)

from crsq.blocks import (
    wave_function,
    hamiltonian,
    discretization,
    rfqhamiltonian,
    radial_func_qrom,
    hamiltonian2
)

from qiskit import QuantumCircuit

dim = 3

def decompose_circuit_to_privmities(circuit):
    while True:
        ops = circuit.count_ops()
        cnames = []
        for op in ops:
            # print(f"op: [{op}]")
            m = re.search('circuit|adder|subtractor|square|sqrt|divide|UMA|MAJ|ucrz', op)
            if m :
                cnames.append(op)
        if len(cnames) == 0:
            return circuit
        # print(f"decompose with {cnames}")
        qc = circuit.decompose(cnames)
        circuit = qc

def rfunc_for_rfq(r):
    return r

def count_gate_rfq(n):
    dq = 1 / 2 ** (n - 1)
    tb = radial_func_qrom.RadialFuncQrom(n, dq, rfunc_for_rfq, verbose=False)
    qc = decompose_circuit_to_privmities(tb.circuit)
    ops = qc.count_ops()
    ops2 = merge_controled_gates(ops)
    print(f"n:{n}  rfq count_ops:{ops2}")
    return qc

def merge_controled_gates(ops):
    """ merge controlled gates cx and ccx with suffixes like _o0 or _o1 into non suffixed ones."""
    odict = {}
    for op in ops:
        m = re.search(r'([a-z]+)_(o\d+)', op)
        if m:
            op0 = m.group(1)
            if op0 not in odict:
                odict[op0] = ops[op]
            else:
                odict[op0] += ops[op]
        else:
            odict[op] = ops[op]
    return odict

# count_gates_and_save_as_csv(5, 8, count_gate_rfq, "rfqrom")
def rfunc_for_rfqh(r):
    return 1/(r+0.5)

def print_summary(n: int, qc: QuantumCircuit, label: str):
    ops = qc.count_ops()
    ops2 = merge_controled_gates(ops)
    width = qc.width()
    print(f"{label} [{n}] width: {width}, count_ops:{ops2}")

def count_gate_arithmetic_elec_potential(n):
    wfr_spec = wave_function.WaveFunctionRegisterSpec(
        dimension=dim,
        num_coordinate_bits=n,
        space_length=32,
        num_electrons=1,
        num_moving_nuclei=0,
        num_stationary_nuclei=1,
    )
    if dim == 2:
        pos = (0,0)
    elif dim == 3:
        pos = (0,0,0)
    ham_spec = hamiltonian.HamiltonianSpec(
        wfr_spec, nuclei_data=[{"charge": 1, "pos": pos}]
    )
    delta_t=1e-3
    disc_spec = discretization.DiscretizationSpec(delta_t)
    epb = hamiltonian.ElectronPotentialBlock(ham_spec, disc_spec)
    qc = decompose_circuit_to_privmities(epb.circuit)
    print_summary(n, qc, "Arithmetic Hamiltonian")
    return qc

def count_gate_rfq(n, label, use_symmetry, use_transpose, use_gray_code):
    wfr_spec = wave_function.WaveFunctionRegisterSpec(
        dimension=dim,
        num_coordinate_bits=n,
        space_length=32,
        num_electrons=1,
        num_moving_nuclei=0,
        num_stationary_nuclei=1,
    )
    ham_spec = hamiltonian.HamiltonianSpec(
        wfr_spec, nuclei_data=[{"charge": 1, "pos": [0, 0]}]
    )
    rfunc = rfunc_for_rfqh
    rfq_spec = rfqhamiltonian.RfqPotentialSpec(
        wfr_spec, rfunc, rfunc,
        use_symmetry=use_symmetry,
        use_transpose=use_transpose,
        use_gray_code=use_gray_code)
    delta_t=1e-3
    disc_spec = discretization.DiscretizationSpec(delta_t)
    repb = rfqhamiltonian.RfqElectronPotentialBlock(rfq_spec, ham_spec, disc_spec)
    qc = decompose_circuit_to_privmities(repb.circuit)
    print_summary(n, qc, label)
    return qc

def count_gate_vsqrom_newton(n):
    wfr_spec = wave_function.WaveFunctionRegisterSpec(
        dimension=dim,
        num_coordinate_bits=n,
        space_length=32,
        num_electrons=1,
        num_moving_nuclei=0,
        num_stationary_nuclei=1,
    )
    ham_spec = hamiltonian.HamiltonianSpec(
        wfr_spec, nuclei_data=[{"charge": 1, "pos": [0, 0]}]
    )
    ham2 = hamiltonian2.InverseSquareRoot(wfr_spec)
    qc = decompose_circuit_to_privmities(ham2.circuit)
    print_summary(n, qc, "VSQROM + Newton-Raphson")
    return qc


def count_gates_and_save_as_csv(N0, N, count_gate_func, label):
    filename = f"output/hamiltonian-gatecount-{label}-{N0}-{N}.csv"
    oplist = []
    for n in range(N0, N+1):
        qc = count_gate_func(n)
        width = qc.width()
        odict = qc.count_ops()
        odict2 = merge_controled_gates(odict)
        odict2["n"] = n
        odict2["width"] = width
        oplist.append(odict2)
    cols0 = [
        "n",
        "width",
        "mcx",
        "ccx",
        "cswap",
        "cx",
        "cp",
        "cu",
        "x",
        "h",
        "u",
        "p",
    ]
    cols = [col for col in cols0 if col in oplist[0]]
    with open(filename, "w") as f:
        f.write(",".join(cols) + "\n")
        for op in oplist:
            f.write(",".join([str(op[col]) for col in cols]) + "\n")

# count_gate_arithmetic_elec_potential(5)

nmin = 5
nmax = 12

count_gates_and_save_as_csv(nmin, nmax, count_gate_vsqrom_newton, "vsqrom-newton")

count_gates_and_save_as_csv(nmin, nmax, count_gate_arithmetic_elec_potential, "arith-elec-potential")

def count_gate_sawtooth_qrom(n):
    return count_gate_rfq(n, "Sawtooth", False, False, False)

def count_gate_sym_qrom(n):
    return count_gate_rfq(n, "Symmetry RFQROM", True, False, False)

def count_gate_sym_trans_qrom(n):
    return count_gate_rfq(n, "Symmetry Transpose RFQROM", True, True, False)

def count_gate_gray_code_qrom(n):
    return count_gate_rfq(n, "Gray code QROM", False, False, True)

# count_gates_and_save_as_csv(nmin, nmax, count_gate_gray_code_qrom, "gray-code-qrom")
# count_gates_and_save_as_csv(nmin, nmax, count_gate_sawtooth_qrom, "sawtooth-qrom")
# count_gates_and_save_as_csv(nmin, nmax, count_gate_sym_qrom, "rfqrom-sym")
# count_gates_and_save_as_csv(nmin, nmax, count_gate_sym_trans_qrom, "rfqrom-sym-trans")


VSQROM + Newton-Raphson [5] width: 215, count_ops:{'ccx': 5872, 'cx': 4698, 'x': 394}
VSQROM + Newton-Raphson [6] width: 250, count_ops:{'ccx': 8060, 'cx': 6382, 'x': 464}
VSQROM + Newton-Raphson [7] width: 285, count_ops:{'ccx': 10592, 'cx': 8320, 'x': 534}
VSQROM + Newton-Raphson [8] width: 320, count_ops:{'ccx': 13468, 'cx': 10518, 'x': 604}
VSQROM + Newton-Raphson [9] width: 355, count_ops:{'ccx': 16688, 'cx': 12940, 'x': 674}
VSQROM + Newton-Raphson [10] width: 390, count_ops:{'ccx': 20252, 'cx': 15662, 'x': 744}
VSQROM + Newton-Raphson [11] width: 425, count_ops:{'ccx': 24160, 'cx': 18586, 'x': 814}
VSQROM + Newton-Raphson [12] width: 460, count_ops:{'ccx': 28412, 'cx': 21816, 'x': 884}


UnboundLocalError: local variable 'pos' referenced before assignment